# Unit 6 Exercise: Finetuning a Pretrained LLM

## Luis Rafael L. Ajoc | BSCS 3A AI

## Task to be performed:
### Text Summarization

## Domain:
### Dialogue Understanding

## LLM to be used:
### DistilBART CNN 12 6

## Install libraries

In [ ]:
!pip install rouge
!pip install bert-score

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import (
    pipeline, 
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer,
    BertTokenizer, 
    BertForMaskedLM, 
    BertModel
)

import accelerate
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics.pairwise import cosine_similarity

from bert_score import BERTScorer
from rouge import Rouge
from rouge_score import rouge_scorer
from sklearn.decomposition import PCA

# from ragas.metrics import summarization_score
# from ragas import evaluate

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "sshleifer/distilbart-cnn-12-6"
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

## Load datasets

In [ ]:
# Load first dataset
ds1 = load_dataset("knkarthick/dialogsum")
ds1 = ds1.remove_columns(['id', 'topic']) # Remove unnecessary columns
print(ds1)

# # Load second dataset
# ds2 = load_dataset("knkarthick/samsum")
# ds2 = ds2.remove_columns(['id'])

# Get training set
# ds1_train = ds1["train"]
# ds2_train = ds2["train"]
# train = concatenate_datasets([ds1_train, ds2_train])

# Get training set
train = ds1["train"]
print(train)

# Get validation set
# ds1_val = ds1["validation"]
# ds2_val = ds2["validation"]
# val = concatenate_datasets([ds1_val, ds2_val])

# Get validation set
val = ds1["validation"]
print(val)

In [ ]:
# assert ds1_train.features.type == ds2_train.features.type

In [ ]:
print(train[0])

## Preprocessing

In [ ]:
# Function to remove '#'
def remove_chars(example):
    example["dialogue"] = example["dialogue"].replace("#", "")
    return example

In [ ]:
# Apply on training and validation dataset
train_updated = train.map(remove_chars)
val_updated = val.map(remove_chars)

In [ ]:
# Check if '#' are removed
print(train_updated[0])
print(val_updated[0])

In [ ]:
# Function for tokenization
def tokenize_inputs(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt').input_ids
    example['labels'] = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt').input_ids

    return example

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
# train_updated = train_updated.filter(lambda example, index: index % 100 == 0, with_indices=True)
train_updated = train_updated.shuffle(seed=42).select(range(len(train_updated) // 2))
train_tokenized = train_updated.map(tokenize_inputs, batched=True)
train_tokenized = train_tokenized.remove_columns(['dialogue', 'summary'])
# val_updated = val_updated.filter(lambda example, index: index % 100 == 0, with_indices=True)
val_updated = val_updated.shuffle(seed=42).select(range(len(val_updated) // 2))
val_tokenized = val_updated.map(tokenize_inputs, batched=True)
val_tokenized = val_tokenized.remove_columns(['dialogue', 'summary'])

In [ ]:
print(train_tokenized)
print(val_tokenized)

In [ ]:
print(train_tokenized.features.type)
print(val_tokenized.features.type)

## Training Configuration

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_weighted": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [ ]:
model.gradient_checkpointing_enable()
model.config.use_cache = False

In [ ]:
# training_args = TrainingArguments(
#     output_dir="./summary-distilbart-12-6",
#     learning_rate=2e-5,
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=4,
#     fp16=True,
#     num_train_epochs=6,
#     weight_decay=0.01,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     logging_steps=5,
#     report_to="none"
# )

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_tokenized,
#     eval_dataset=val_tokenized,
#     processing_class=tokenizer,
#     data_collator=data_collator,
#     compute_metrics=compute_metrics,
# )

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
trainer.save_model("./summary-distilbart-12-6-output")
tokenizer.save_pretrained("./summary-distilbart-12-6-output")

In [ ]:
# Function to generate summary
def generate_summary(input, llm):
    input_prompt = f"""
                    Summarize the following conversations.
                    
                    {input}
                    
                    Summary:
                    """
    input_ids = loaded_tokenizer(input_prompt, return_tensors='pt')
    summary_ids = llm.generate(
        input_ids["input_ids"], 
        min_length=30, 
        max_length=200,
        num_beams=4,
        early_stopping=True
    )
    output = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return output

## Evaluation

In [ ]:
model_path = "/kaggle/working/summary-distilbart-12-6-output"
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

In [ ]:
text = """
Beatrice: I am in town, shopping. They have nice scarfs in the shop next to the church. Do you want one?
Leo: No, thanks
Beatrice: But you don't have a scarf.
Leo: Because I don't need it.
Beatrice: Last winter you had a cold all the time. A scarf could help.
Leo: I don't like them.
Beatrice: Actually, I don't care. You will get a scarf.
Leo: How understanding of you!
Beatrice: You were complaining the whole winter that you're going to die. I've had enough.
Leo: Eh.
"""

label = "Beatrice wants to buy Leo a scarf, but he doesn't like scarves. She cares about his health and will buy him a scarf no matter his opinion."

output = generate_summary(text, llm=loaded_model)

print("---ORIGINAL TEXT---")
print(text)
print("-------------------")
print("-GENERATED SUMMARY-")
print(output)

In [ ]:
bertscorer = BERTScorer(model_type='bert-base-uncased')
P, R, F1 = bertscorer.score([text], [output])
print(f"BERTScore Precision: {P.mean():.4f}, Recall: {R.mean():.4f}, F1: {F1.mean():.4f}")

In [ ]:
# ROUGE calculation
rougescorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rougescores = rougescorer.score(text, output)
print(f"ROUGE-1 Precision: {rougescores['rouge1'].precision:.4f}, Recall: {rougescores['rouge1'].recall:.4f}, F1: {rougescores['rouge1'].fmeasure:.4f}")
print(f"ROUGE-2 Precision: {rougescores['rouge2'].precision:.4f}, Recall: {rougescores['rouge2'].recall:.4f}, F1: {rougescores['rouge2'].fmeasure:.4f}")
print(f"ROUGE-L Precision: {rougescores['rougeL'].precision:.4f}, Recall: {rougescores['rougeL'].recall:.4f}, F1: {rougescores['rougeL'].fmeasure:.4f}")

In [ ]:
words = [
    "meeting", "schedule", "tomorrow", "confirm", "project",
    "update", "deadline", "discuss", "report", "email",
    "client", "call", "plan", "review", "team",
    "message", "task", "lunch", "office", "follow-up"
]

In [ ]:
def get_embedding(text):
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    
    # Mean pooling
    embedding = outputs.encoder_last_hidden_state.mean(dim=1)
    return embedding.squeeze().numpy()

embeddings = [get_embedding(word) for word in words]

In [ ]:
X = np.array(embeddings)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

In [ ]:
plt.figure(figsize=(8,6))

for i, word in enumerate(words):
    x, y = X_pca[i]
    plt.scatter(x, y)
    plt.annotate(word, (x, y))

plt.title("PCA Visualization")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()

In [ ]:
# def apply_plot_pca(words, model):
#     filtered_words = [w for w in words if w in model.wv]
#     vectors = np.array([model.wv[w] for w in filtered_words])

#     pca = PCA(n_components=2)
#     reduced = pca.fit_transform(vectors)

#     plt.figure(figsize=(10, 7))
#     plt.scatter(reduced[:, 0], reduced[:, 1])

#     for i, word in enumerate(filtered_words):
#         plt.annotate(
#             word,
#             (reduced[i, 0], reduced[i, 1]),
#             xytext=(5, 5),
#             textcoords="offset points",
#             fontsize=9
#         )

#     plt.title("PCA Visualization")
#     plt.xlabel("Principal Component 1")
#     plt.ylabel("Principal Component 2")
#     plt.grid(True)

#     plt.show()

In [ ]:
# apply_plot_pca(words, loaded_model)